Production-grade multimodal embedding validation pipeline.

Responsibilities:
- embedding diagnostics
- cross-modal alignment validation
- outlier detection
- retrieval health analysis
- cluster diagnostics
- semantic drift analysis
- validated artifact generation

In [1]:
!pip install faiss-gpu-cu12
!pip install -q umap-learn
!pip install -q scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 39.4 MB/s eta 0:00:00


In [2]:
import gc
import faiss
import warnings
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import normalize

from sklearn.decomposition import PCA

import umap.umap_ as umap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

ARTIFACT_DIR = (
    "/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1"
)

2026-05-12 12:47:32.814578: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778590053.027694      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778590053.091801      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778590053.614518      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778590053.614567      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778590053.614570      23 computation_placer.cc:177] computation placer alr

In [3]:
import glob

artifact_files = glob.glob(
    f"{ARTIFACT_DIR}/**/*",
    recursive=True
)

for f in artifact_files:
    print(f)

/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/fusion_embeddings.npy
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/downloaded_images
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/normalized_tensors
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/multimodal_faiss.index
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/caption_embeddings.npy
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/retrieval_ready_df.parquet
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/downloaded_images/B09GHFT3M9_4302.jpg
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/downloaded_images/B0CQY7PMMK_313.jpg
/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/downloaded_images/B01JZ0XPDE_1110.jpg
/kaggle/input/notebooks/ha

In [4]:
caption_embedding_files = glob.glob(
    f"{ARTIFACT_DIR}/caption_embeddings.npy"
)

fusion_embedding_files = glob.glob(
    f"{ARTIFACT_DIR}/fusion_embeddings.npy"
)

print("CAPTION FILES:\n")

for f in caption_embedding_files[:10]:
    print(f)

print("\nFUSION FILES:\n")

for f in fusion_embedding_files[:10]:
    print(f)

CAPTION FILES:

/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/caption_embeddings.npy

FUSION FILES:

/kaggle/input/notebooks/hanafudaearring/media-resolution-captioning/artifacts/v1/fusion_embeddings.npy


In [5]:
import os
import torch

def load_embedding_artifact(path):

    # NUMPY
    if path.endswith(".npy"):
        return np.load(path)

    # TORCH
    elif path.endswith(".pt"):
        return torch.load(path)

    # PARQUET
    elif path.endswith(".parquet"):
        return pd.read_parquet(path)

    else:
        return None

In [6]:
fusion_embedding_path = (
    f"{ARTIFACT_DIR}/fusion_embeddings.npy"
)

fusion_embeddings = load_embedding_artifact(
    fusion_embedding_path
)

fusion_embeddings.shape

(4343, 512)

In [7]:
caption_embedding_path = (
    f"{ARTIFACT_DIR}/caption_embeddings.npy"
)

caption_embeddings = load_embedding_artifact(
    caption_embedding_path
)

caption_embeddings.shape

(4343, 512)

In [8]:
from sentence_transformers import (
    SentenceTransformer
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

clip_model = SentenceTransformer(
    "clip-ViT-B-32",
    device=DEVICE
)

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [9]:
retrieval_df = pd.read_parquet(

    f"{ARTIFACT_DIR}/retrieval_ready_df.parquet"
)

print(retrieval_df.shape)

retrieval_df.head()

(4343, 24)


,image_path,width,height,aspect_ratio,megapixels,is_low_resolution,is_extreme_aspect_ratio,blur_score,is_blurry,brightness_score,is_dark_image,contrast_score,quality_score,is_low_quality,phash,duplicate_count,is_duplicate,tensor_path,caption,caption_length,is_empty_caption,is_repetitive_caption,image_caption_similarity,is_low_alignment
0,/kaggle/working/artifacts/v1//downloaded_image...,400,500,0.8,0.202506,False,False,0.005658,False,0.741763,False,0.580375,0.326877,True,ea48d58e8aadb991,1,False,/kaggle/working/artifacts/v1//normalized_tenso...,men ' s polo shirt,5,False,False,0.681957,False
1,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.253234,False,False,0.020590,False,0.764080,False,0.694872,0.373938,False,ee9c952592626b8d,1,False,/kaggle/working/artifacts/v1//normalized_tenso...,a man wearing a black polo shirt with a smile ...,12,False,False,0.486107,False
2,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.253234,False,False,0.013650,False,0.724196,False,0.669775,0.358860,False,faecc059a53781ca,1,False,/kaggle/working/artifacts/v1//normalized_tenso...,a man wearing a grey polo shirt and white pants,10,False,False,0.410553,False
3,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.253234,False,False,0.009926,False,0.651542,False,0.641367,0.337530,True,afe6c458991d8749,1,False,/kaggle/working/artifacts/v1//normalized_tenso...,a man walking with a golf bag,7,False,False,0.590835,False
4,/kaggle/working/artifacts/v1//downloaded_image...,500,500,1.0,0.253234,False,False,0.006821,False,0.863412,False,0.557901,0.362279,False,a6cea6cf3938b083,1,False,/kaggle/working/artifacts/v1//normalized_tenso...,a man standing on the beach with his hands in ...,12,False,False,0.604645,False


In [10]:
image_paths = retrieval_df[
    "image_path"
].tolist()

image_embeddings = clip_model.encode(

    image_paths,

    batch_size=32,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

Batches:   0%|          | 0/136 [00:00<?, ?it/s]

In [11]:

ARTIFACT_2_DIR = "/kaggle/working/artifacts_2/v2"

from pathlib import Path

Path(ARTIFACT_2_DIR).mkdir(
    parents=True,
    exist_ok=True
)


import os

print(os.path.exists(ARTIFACT_2_DIR))

True


In [12]:
np.save(

    f"{ARTIFACT_2_DIR}/image_embeddings.npy",

    image_embeddings
)

In [13]:
print("IMAGE:", image_embeddings.shape)

print("CAPTION:", caption_embeddings.shape)

print("FUSION:", fusion_embeddings.shape)

print("DF:", retrieval_df.shape)

IMAGE: (4343, 512)
CAPTION: (4343, 512)
FUSION: (4343, 512)
DF: (4343, 24)


In [14]:
assert len(retrieval_df) == len(image_embeddings)

assert len(retrieval_df) == len(caption_embeddings)

assert len(retrieval_df) == len(fusion_embeddings)

print("PIPELINE CONSISTENT")

PIPELINE CONSISTENT


In [15]:
def validate_embeddings(embeddings):

    nan_count = np.isnan(
        embeddings
    ).sum()

    inf_count = np.isinf(
        embeddings
    ).sum()

    return {

        "nan_count": int(nan_count),

        "inf_count": int(inf_count)
    }

In [16]:
validate_embeddings(
    image_embeddings
)

{'nan_count': 0, 'inf_count': 0}

In [17]:
validate_embeddings(
    caption_embeddings
)

{'nan_count': 0, 'inf_count': 0}

In [18]:
validate_embeddings(
    fusion_embeddings
)

{'nan_count': 0, 'inf_count': 0}

In [19]:
def compute_norms(embeddings):

    return np.linalg.norm(
        embeddings,
        axis=1
    )

In [20]:
image_norms = compute_norms(
    image_embeddings
)

pd.Series(image_norms).describe()

count    4.343000e+03
mean     1.000000e+00
std      4.839161e-08
min      9.999998e-01
25%      9.999999e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [21]:
caption_norms = compute_norms(
    caption_embeddings
)

pd.Series(caption_norms).describe()

count    4.343000e+03
mean     1.000000e+00
std      4.104529e-08
min      9.999998e-01
25%      9.999999e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [22]:
fusion_norms = compute_norms(
    fusion_embeddings
)

pd.Series(fusion_norms).describe()

count    4.343000e+03
mean     1.000000e+00
std      4.931279e-08
min      9.999998e-01
25%      9.999999e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [23]:
retrieval_df["bad_embedding_norm"] = (

    np.abs(
        fusion_norms - 1.0
    ) > 0.01
)

In [24]:
print(
    "BAD NORMS:",
    retrieval_df[
        "bad_embedding_norm"
    ].sum()
)

BAD NORMS: 0


In [25]:
alignment_scores = np.sum(

    image_embeddings
    *
    caption_embeddings,

    axis=1
)

In [26]:
retrieval_df[
    "embedding_alignment_score"
] = alignment_scores

In [27]:
retrieval_df[
    "embedding_alignment_score"
].describe()

count    4343.000000
mean        0.563847
std         0.127825
min         0.114647
25%         0.486813
50%         0.572338
75%         0.649745
max         0.874453
Name: embedding_alignment_score, dtype: float64

In [28]:
retrieval_df[
    "low_alignment_flag"
] = (

    retrieval_df[
        "embedding_alignment_score"
    ] < 0.18
)

In [29]:
print(
    "LOW ALIGNMENT:",
    retrieval_df[
        "low_alignment_flag"
    ].sum()
)

LOW ALIGNMENT: 10


In [30]:
from sklearn.neighbors import (
    LocalOutlierFactor
)

In [31]:
lof = LocalOutlierFactor(

    n_neighbors=20,

    contamination=0.03
)

In [32]:
outlier_labels = lof.fit_predict(
    fusion_embeddings
)

In [33]:
retrieval_df[
    "embedding_outlier"
] = (

    outlier_labels == -1
)

In [34]:
print(
    "OUTLIER EMBEDDINGS:",
    retrieval_df[
        "embedding_outlier"
    ].sum()
)

OUTLIER EMBEDDINGS: 131


In [35]:
retrieval_df[

    retrieval_df[
        "embedding_outlier"
    ]

][[
    "caption",
    "quality_score",
    "embedding_alignment_score"
]].head(20)

,caption,quality_score,embedding_alignment_score
16,a logo for a fire company,0.204016,0.624209
40,a deer is seen in the dark,0.146711,0.683972
48,the beatles - the beatles,0.433981,0.706728
78,a col of golf players,0.291519,0.780118
131,a brush with no bre and no bre,0.310894,0.821173
173,a man wearing a white v - neck shirt with the ...,0.373320,0.601341
226,a man in a white shirt and black pants is walk...,0.230164,0.555920
265,a man in a tan shirt and white pants stands by...,0.286316,0.494839
286,a green and white logo with the word ' s ',0.304961,0.542667
295,a bathing bathing bathing bathing bathing bath...,0.277336,0.757149


In [36]:
from sklearn.cluster import (
    KMeans
)

NUM_CLUSTERS = 25

kmeans = KMeans(

    n_clusters=NUM_CLUSTERS,

    random_state=42
)

cluster_labels = kmeans.fit_predict(
    fusion_embeddings
)

retrieval_df[
    "semantic_cluster"
] = cluster_labels

retrieval_df[
    "semantic_cluster"
].value_counts()

semantic_cluster
7     583
21    248
22    239
17    238
19    211
8     201
3     195
4     192
20    188
0     181
12    171
13    171
24    159
14    155
6     151
5     149
16    143
10    137
1     121
11    103
23     99
9      91
18     83
15     68
2      66
Name: count, dtype: int64

In [37]:
retrieval_df.groupby(
    "semantic_cluster"
)["caption"].apply(list).head()

semantic_cluster
0    [the north face men ' s long sleeve plaid shir...
1    [a pink and white striped polo shirt, a man in...
2    [a white and gold cigarette case with the word...
3    [a man wearing a black polo shirt with a smile...
4    [a diagram of the sizes of the person ' s shoe...
Name: caption, dtype: object

In [38]:
retrieval_df[
    "normalized_alignment"
] = (

    retrieval_df[
        "embedding_alignment_score"
    ]
    -
    retrieval_df[
        "embedding_alignment_score"
    ].min()

) / (

    retrieval_df[
        "embedding_alignment_score"
    ].max()

    -
    retrieval_df[
        "embedding_alignment_score"
    ].min()
)

In [39]:
retrieval_df[
    "embedding_health_score"
] = (

    0.5
    *
    retrieval_df[
        "normalized_alignment"
    ]

    +

    0.3
    *
    retrieval_df[
        "quality_score"
    ]

    +

    0.2
    *
    (~retrieval_df[
        "embedding_outlier"
    ]).astype(int)
)

In [40]:
retrieval_df[
    "embedding_health_score"
].describe()

count    4343.000000
mean        0.586821
std         0.090922
min         0.058999
25%         0.534449
50%         0.596708
75%         0.645672
max         0.826729
Name: embedding_health_score, dtype: float64

In [41]:
retrieval_df[
    "low_embedding_health"
] = (

    retrieval_df[
        "embedding_health_score"
    ] < 0.4
)

In [42]:
print(
    "LOW HEALTH EMBEDDINGS:",
    retrieval_df[
        "low_embedding_health"
    ].sum()
)

LOW HEALTH EMBEDDINGS: 131


In [43]:
retrieval_df[

    retrieval_df[
        "low_embedding_health"
    ]

][[
    "caption",
    "quality_score",
    "embedding_alignment_score",
    "embedding_health_score"
]].head(20)

,caption,quality_score,embedding_alignment_score,embedding_health_score
16,a logo for a fire company,0.204016,0.624209,0.396529
144,a man in a black shirt and plaid shorts standi...,0.296712,0.262600,0.386376
226,a man in a white shirt and black pants is walk...,0.230164,0.555920,0.359435
253,a man standing on the beach wearing a blue t s...,0.257829,0.295164,0.396140
265,a man in a tan shirt and white pants stands by...,0.286316,0.494839,0.336085
286,a green and white logo with the word ' s ',0.304961,0.542667,0.373152
292,a green camouflage polo shirt with a black and...,0.319672,0.238968,0.377712
309,a man in a black shirt and khaki pants stands ...,0.197007,0.418402,0.258992
323,a man in a black shirt and plaid shorts standi...,0.296712,0.257916,0.383293
382,a man leaning against a wall wearing a green t...,0.308186,0.265175,0.391513


In [44]:
VALIDATION_DIR = (
    "/kaggle/working/artifacts_2/v2"
)

Path(VALIDATION_DIR).mkdir(
    parents=True,
    exist_ok=True
)

In [45]:
retrieval_df.to_parquet(

    f"{VALIDATION_DIR}/validated_embeddings_df.parquet",

    index=False
)

In [46]:
np.save(

    f"{VALIDATION_DIR}/validated_fusion_embeddings.npy",

    fusion_embeddings
)

In [47]:
validation_summary = {

    "total_embeddings":
        int(len(retrieval_df)),

    "low_alignment":
        int(
            retrieval_df[
                "low_alignment_flag"
            ].sum()
        ),

    "outliers":
        int(
            retrieval_df[
                "embedding_outlier"
            ].sum()
        ),

    "low_health":
        int(
            retrieval_df[
                "low_embedding_health"
            ].sum()
        )
}

In [48]:
import json

with open(

    f"{VALIDATION_DIR}/embedding_health_report.json",

    "w"

) as f:

    json.dump(
        validation_summary,
        f,
        indent=4
    )

In [49]:
print("=" * 60)

print("EMBEDDING VALIDATION COMPLETE")

print("=" * 60)

print(
    "TOTAL EMBEDDINGS:",
    len(retrieval_df)
)

print(
    "LOW ALIGNMENT:",
    retrieval_df[
        "low_alignment_flag"
    ].sum()
)

print(
    "OUTLIERS:",
    retrieval_df[
        "embedding_outlier"
    ].sum()
)

print(
    "LOW HEALTH:",
    retrieval_df[
        "low_embedding_health"
    ].sum()
)

print("=" * 60)

EMBEDDING VALIDATION COMPLETE
TOTAL EMBEDDINGS: 4343
LOW ALIGNMENT: 10
OUTLIERS: 131
LOW HEALTH: 131
